---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: Web Analytics

### 📋 **Topic**: Scraping NOAA news using Beautiful Soup

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/wa/blob/main/assets/before-you-begin.md)

# 🌐 0. Introduction

This notebook details how we can scrape news from [NOAA](https://www.noaa.gov/news), the National Oceanic and Atmospheric Administration.

We will first get the title and URL of each article. Then we will get several pages of articles. At the end, the challenge is to also get their dates and categories, and save everything in a file.

As always, we will begin by importing our libraries.

Let's also create
- `data`: an empty list, where we will be appending our content
- `my_headers`: a dictionary for any additional request headers. We will leave it empty and use the normal defaults from `requests`.

We will only get five pages, and pause between requests. If a website blocks a request, don't keep hammering it! A request header does not turn Python into a browser or complete a browser-verification check.

**If NOAA blocks the request:** set `use_saved_pages = True` in the setup cell below, then run the notebook again from the top. This reads five NOAA news pages saved on September 28, 2026, instead of downloading live pages. The scraping steps are the same, but those articles are a snapshot, not today's news. The code will tell you when it is using saved HTML.


In [ ]:
# 1. importing useful libraries

import requests # For making HTTP requests to the web.
import time     # To introduce pauses in our code, ensuring we don't overwhelm the server.
from bs4 import BeautifulSoup # A popular library to parse and navigate HTML content
from pathlib import Path # For the optional saved-page files.
from urllib.parse import urljoin # To turn relative links into full URLs.

# create an empty list to store our scraped data
data = []

# We don't need any extra headers here. Use the normal requests defaults.
my_headers = {}

# Normally, get the live NOAA pages. Change this to True only to use the saved copies.
use_saved_pages = False

# Locate the saved pages whether this notebook starts in wa or wa/scripts.
course_folder = Path.cwd()
if course_folder.name in ('scripts', 'homework'):
    course_folder = course_folder.parent
saved_pages_folder = course_folder / 'assets' / 'noaa-news'


---
# 📕 1. Fetching the Webpage

First, we will try to get the page up to five times:
- ✅ **Success**: if we are successful, we will immediately exit the loop
- 🚫 **Failure**: if we are unsuccessful, the code will print a message telling us what went wrong

`timeout=15` means we will not wait forever for a response. `raise_for_status()` checks whether the server actually sent us a successful response. Without this check, we might mistake an error page for the news page!

We will retry request errors, but stop if the server says we are not allowed to access the page (`401` or `403`), or the page does not exist (`404`). At the end, we will tell the user whether we got the page or not. If we didn't, we stop here rather than try to parse an error page.

An empty response is not useful either, even if its status is not an error. We check for that too. If the live page is blocked or empty, use the saved-page option described above; don't keep sending requests to a blocked page.


In [ ]:
# Define the target URL
page = 'https://www.noaa.gov/news'
snapshot_number = 0

# Initialize the source content to None
src = None

if use_saved_pages:
    snapshot_file = saved_pages_folder / f'page-{snapshot_number}.html'
    src = snapshot_file.read_bytes()
    print(f'Using saved NOAA HTML from September 28, 2026: {snapshot_file.name} (not live news)')
else:
    # Try to get the page up to 5 times
    for i in range(5):
        try:
            response = requests.get(page, headers=my_headers, timeout=15)
            response.raise_for_status()
            if response.status_code != 200 or not response.content:
                raise requests.HTTPError(f'Expected a nonempty HTTP 200 page; got HTTP {response.status_code}.', response=response)
            src = response.content
            break
        except requests.RequestException as error:
            print(f'Failed attempt # {i + 1}: {error}')
            # Don't retry a page that is blocked or does not exist.
            if error.response is not None and error.response.status_code in (202, 401, 403, 404):
                break
            if i < 4:
                time.sleep(2)


if src:
    if not use_saved_pages:
        print(f'Successfully got page: {page}')
else:
    raise RuntimeError(f'Could not get page: {page}. Check the error above, or set use_saved_pages = True and restart the notebook.')


Try to get a page that does not exist, e.g. `https://www.noaa.gov/this-page-does-not-exist`, and see what happens! Change `page` in the cell above and run it again. Then put the news URL back before continuing.


---
# 🔢 2. Encoding

When we fetch data from the web using libraries like requests, the data we get can be in various formats such as HTML, JSON, images, etc. This raw data is sent over the internet in the form of bytes. Therefore, when we receive the data, it's in its raw byte form. 

Bytes are represented in Python as a `bytes` object.
- The bytes object in Python is an immutable sequence of bytes used to represent raw, binary data.
- "Bytes" are the smallest addressable unit in a computer and can represent a wide variety of data, including text, images, audio, and more.

`bytes` are different from `strings`
-  Strings in Python represent text, and each character in a string corresponds to a specific symbol. When we want to store or transmit this text, it needs to be encoded into bytes, which is where encodings like ASCII, utf-8, and others come into play. 
-  Each encoding has its own way of mapping characters to sequences of bytes.
-  Strings are sequences of characters, while bytes are sequences of bytes.
-  Conversion between strings and bytes is done via encoding (to bytes) and decoding (to string):

To decode bytes to strings, we need to know how the bytes were encoded.
- The server tells us the encoding of the data it sends us in the response headers.
- requests also comes with a built-in encoding detector, which guesses the encoding of the content it receives

In [ ]:
# The full HTML is long! Let's just look at the first 500 bytes.
print(src[:500])
if use_saved_pages:
    print('Saved HTML encoding: UTF-8')
else:
    print('Encoding as per headers:', response.encoding)
    print('Apparent encoding:', response.apparent_encoding)


---

# 🍜 3. Using Beautiful Soup to parse the information

Beautiful Soup is a Python library designed for web scraping purposes to pull the data out of HTML and XML files. It creates parse trees from these files that are helpful to extract the data easily.

## 3.1 Parsing the HTML as a Beautiful Soup object

Before we parse the content, let's turn our bytes into text. NOAA's page uses UTF-8, so we will decode it with `utf-8`. This keeps characters such as curly apostrophes and accented letters. ASCII cannot represent all of these characters.


In [ ]:
# Turn the UTF-8 bytes into a string, then create a BeautifulSoup object.
html = src.decode('utf-8')
soup = BeautifulSoup(html, 'html.parser')

# confirm that you have successfully created a BeautifulSoup object
type(soup)


Why `decode('utf-8')`?
- **`decode('utf-8')`**: This converts the fetched bytes into a string using the UTF-8 encoding.
- We are not using `'ignore'`: that option silently drops bytes it cannot decode. If the encoding is wrong, we want to notice it rather than lose part of the text!

The `BeautifulSoup()` constructor needs two arguments:
- **The HTML (or XML) content** that we are going to parse.
- **The parser library name.** In this case, we're using `html.parser`, which is a nice built-in parser. There are other parsers, such as `html5lib` and `lxml`. Often, the `lxml` parser is preferred for its speed.


## 3.2 Parse Tree

A parse tree, often referred to as a *parsing tree* or *syntax tree*, represents syntactic constructs in source code or any structured text. It's utilized to portray the code or content's structure in a hierarchical format.

For HTML and XML documents, the parse tree denotes the structure of the document. Each tag, attribute, and piece of text stands as a node in this tree.

### Key Concepts:

- **Node**: Each unique part of the tree, such as an HTML tag, is termed a node. For instance, in an HTML document like `<title>My Web Page</title>`, both the `<title>` tag and the text "My Web Page" are nodes.

- **Root Node**: This is the highest node in the tree. In the context of an HTML document, the root node is typically represented by the `<html>` tag.

- **Child Nodes**: These are nodes that stem from another node. For HTML, tags and content nested within another tag are its child nodes. As an example, in `<body><p>Text</p></body>`, `<p>` is a child node of `<body>`.

- **Parent Node**: The opposite of child nodes. If node A is a descendant of node B, node B is the progenitor of node A.

- **Sibling Nodes**: These nodes share the same parent. If two tags or textual pieces are housed within the same parent tag, they are acknowledged as siblings.

### Why is the Parse Tree Important?

- **Navigation**: With the HTML or XML content transformed into a parse tree, navigation becomes possible. This empowers users to discover specific document segments, traverse the structure in various directions, and pull out the desired data.

- **Search**: Specific tags, classes, IDs, or other attributes can be located. This capability is a cornerstone in web scraping.

- **Modification**: Content can be altered through the parse tree. This includes the addition, removal, or alteration of tags and attributes.

- **Extraction**: Once the desired data is pinpointed, structured extraction becomes feasible.

### Example
Here's some HTML
```
<html>
    <head>
        <title>My Web Page</title>
    </head>
    <body>
        <p class="intro">Welcome to my web page!</p>
        <p>Here's some more text.</p>
    </body>
</html>
```

Parse Tree Structure:
- `<html>` (Root Node)
  - `<head>` (Child of `<html>`)
    - `<title>` (Child of `<head>`)
      - "My Web Page" (Text node, child of `<title>`)
  - `<body>` (Child of `<html>`, sibling of `<head>`)
    - `<p class="intro">` (Child of <body>)
      - "Welcome to my web page!" (Text node, child of first `<p>`)
    - `<p>` (Second `<p>` tag, child of `<body>`, sibling of the first `<p>`)
      - "Here's some more text." (Text node, child of second `<p>`)

The representation above is a basic idea of how the parse tree would look for the given HTML.

### Beautiful Soup and the Parse Tree:

When Beautiful Soup is employed to dissect an HTML or XML document, it creates an in-memory parse tree from the page source code. This grants the ability to engage with the page's architecture, navigate, seek out, and alter its components. Functions such as 
- `find()`, 
- `find_all()`, 
- `parent`, 
- `children`, 
- `next_sibling`, 
- `previous_sibling`, 

and more, facilitate movement and interaction within this tree.


## 3.3 Using find_all with Beautiful Soup
The `find_all` method is one of the most commonly used methods in Beautiful Soup to search the parse tree. It returns all the matching tags found in the document, in the form of a list.

In [ ]:
links = soup.find_all("a")
print(len(links))
links[0]


In the above line, `soup.find_all("a")` looks for all `<a>` tags, commonly used for hyperlinks, in the parsed HTML document.

We could try to find links with a regular expression, as we did before. But that is not the same as parsing HTML: a pattern can accidentally grab several links at once, or miss links spread over multiple lines. Beautiful Soup understands the tags and their structure, which makes our job much easier here.


In [ ]:
element = links[0]


In [ ]:
element.parent

In [ ]:
for k in element.children:
    print(k)

Let's do another example:

In [ ]:
element = links[-3]
print(f"Here is the element we grabbed: {element}")

In [ ]:
print("Here is the text of the element:")
print(element.text)

In [ ]:
print("Here are the attributes of the element:")
print(element.attrs)

In [ ]:
print("Here is the parent of the element:")
print(element.parent)

In [ ]:
print("Here are the children of the element:")
for k in element.parent.children:
    print(f"- {k}")

---
## 🔍 4. Finding Data Location using the "Inspect" tool

To extract the correct information, we first need to find where it lives in the webpage. For each NOAA article, we want its headline and link. Later, we will also get its date and "Focus areas" categories.

Here is the [NOAA news page](https://www.noaa.gov/news). The headlines will change, but look for the same repeated article blocks:

![NOAA news page with red callouts for the headline and link, date, and Focus areas](../assets/noaa-1-annotated.png)

Using your browser's "inspect" feature makes the scraping process easier:

1. **Open Inspect**: Right-click on a headline and select "Inspect".
2. **Element Selector**: Click the "Select an element" tool.
3. **Move around the page**: Hover over the webpage elements. The corresponding HTML will be highlighted. Click when you've found the element you want.

Here is one article. Right-click its headline and choose "Inspect". Find the headline's `<a>` tag, then move up to the `<article>` tag that contains it. Notice that the headline, date, categories, image, and topic tags belong to the same article:

![One NOAA article outlined in red, with numbered instructions to inspect the headline and find its article container](../assets/noaa-2-annotated.png)

Using the inspect tool allows you to find the HTML segments containing the data you want to access. Make sure you inspect the headline, not the image or a category link!

### 🌍 **Google Chrome**:
1. **Right-click** on an element.
2. Select **"Inspect"** from the context menu.
3. Alternatively, use the shortcut **Ctrl + Shift + I** (Windows/Linux) or **Cmd + Option + I** (Mac).

### 🦊 **Mozilla Firefox**:
1. **Right-click** on an element.
2. Choose **"Inspect Element"** from the dropdown.
3. Shortcut: **Ctrl + Shift + C** (Windows/Linux) or **Cmd + Option + C** (Mac).

### 🌐 **Microsoft Edge**:
1. **Right-click** on the desired item.
2. Opt for **"Inspect Element"**.
3. You can also use **F12** or **Ctrl + Shift + I** to directly open the developer tools.

### 🍏 **Safari**:
(Note: You might need to enable the "Develop" menu first from Settings > Advanced.)
1. **Right-click** on the element (if you have a Mac with a single button mouse, Ctrl + click).
2. Choose **"Inspect Element"**.
3. Shortcut: **Cmd + Option + C**.


---
# 🎯 5. Getting what we want

## 5.1 Finding a tag

Next, we need to find the tag that "contains" the information we want.

Inspecting NOAA's HTML, we find an `<article>` tag with the class `node-teaser`. Each of these tags contains the information for one news article:

1. A `<div class="title">`, which contains the headline's `<a>` tag. That link gives us both the headline text and its URL.
2. A `<time>` tag with the publication date.
3. A `<div class="tags focus-areas">` with links to its categories.

By saying "contain", we mean that this information is found after the `<article>` tag opens, and before it closes.
- Alternatively, we are looking inside the `<article>` tag, at its children and their children.

This is easy to see in the "inspect view", because everything that an element contains has more indent than that element!!

Here is a simplified view of the relevant HTML. NOAA has more wrapper tags in between, but these are the parts we want:

![Simplified NOAA article HTML with red boxes and arrows labeling the article container, headline link, date, and Focus areas](../assets/noaa-3-annotated.svg)

To find all news article tags, we simply ask Beautiful Soup to look for this tag and class in the `soup` variable---that contains the HTML.


In [ ]:
# find all article tags
entries = soup.find_all('article')

# only keep the article tags that have the class "node-teaser"
filtered_entries = [entry for entry in entries if 'node-teaser' in entry.get('class', [])]

# how many entries did we find?
print(f'We found {len(filtered_entries)} entries')
if not filtered_entries:
    raise RuntimeError('We found no news articles. Check that we got the news page, not an error page.')


Let's take a look at the first entry:

In [ ]:
print(filtered_entries[0].prettify())

In [ ]:
# And the last entry. Using -1 still works if the page has fewer articles.
print(filtered_entries[-1].prettify())


You can take a look at the attributes of that HTML tag, and verify that these are the tags that we want.


### Note
Let's say you have the following HTML:
```html
<div>
    <a href="#">Link 1</a>
    <div>
        <a href="#">Nested Link 1</a>
        <a href="#">Nested Link 2</a>
    </div>
</div>
```
If you use `soup.find_all("a")`, it will return all three `<a>` tags: 
- ```text<a href="#">Link 1</a>```
- ```text<a href="#">Nested Link 1</a>```
- ```text<a href="#">Nested Link 2</a>```

Even though Nested Link 1, and Nested Link 2 are inside a nested `<div>`, BS returns them as separate matches.


Alright, now that we have all the news articles, let's grab the correct information for each article.
We need to grab two kinds of information:

- the headline text
- the URL of the article

There are several `<a>` tags in each article: links around images, category links, topic links, and the headline link. So we will first find the `<div class="title">`, and then find the `<a>` inside it. We don't want to accidentally grab an image link with no text!


## 5.3 Getting the text
Let's first do this with the first article, and then do it for every article using a for loop.


In [ ]:
entry = filtered_entries[0]
print(entry.prettify())
print(type(entry))


In [ ]:
# first grab the <div class="title"> and everything it contains from the first article
# to do so, simply use Beautiful Soup's "find" functionality
# (because there is only one matching title div here, it will return that one)
entry_text = entry.find('div', class_='title').get_text()
print(entry_text)


In [ ]:
# We could also grab the text of the <a> tag inside the title div.
entry_text = entry.find('div', class_='title').find('a').get_text()
print(entry_text)


In [ ]:
# strip unnecessary whitespace using the string's strip() method
entry_text = entry.find('div', class_='title').find('a').get_text().strip()
print(entry_text)


## 5.4 Getting the URL

The headline link has an `href` attribute. On NOAA, it is often a **relative URL**, like `/news/some-article`, rather than a full URL starting with `https://`.

`urljoin()` combines it with the page URL to give us a full link, like `https://www.noaa.gov/news/some-article`. It also works if the link is already a full URL.


In [ ]:
# find the headline's <a> tag inside <div class="title">
entry_url = entry.find('div', class_='title').find('a')
# keep only the value of the href attribute
entry_url = entry_url.attrs.get('href')
print(f'Relative link: {entry_url}')

# turn the relative link into a full URL
entry_url = urljoin(page, entry_url)
print(entry_url)


In [ ]:
print(f'url   = {entry_url}')
print(f'text  = {entry_text}')


## 5.5 Getting information for all articles
Now to scrape all articles from the webpage, simply use a for loop!

In [ ]:
data = []

entries = soup.find_all('article', class_='node-teaser')

for entry in entries:
    # find the title link, grab its text, strip() it
    title_link = entry.find('div', class_='title').find('a')
    entry_text = title_link.get_text().strip()

    # and get the full URL too
    entry_url = urljoin(page, title_link.attrs.get('href'))

    # add all the info to the data list
    data.append([entry_url, entry_text])

for i, entry in enumerate(data):
    print(f'Entry {i + 1}:')
    print(f'- URL:  {entry[0]}')
    print(f'- Text: {entry[1]}')


## 5.6 Scraping all articles in webpage with failsafes
Let's now see how to deal with cases where we search for elements that do not exist


In [ ]:
data = []

entries = soup.find_all('article', class_='node-teaser')

for entry in entries:
    entry_text = None
    entry_url = None

    # find the title container first, then the link inside it
    found_element = entry.find('div', class_='title')
    if found_element:
        found_element = found_element.find('a')
        if found_element:
            entry_text = found_element.get_text().strip()
            href = found_element.attrs.get('href')
            if href:
                entry_url = urljoin(page, href)

    # add all the info to the data list
    data.append([entry_url, entry_text])

for i, entry in enumerate(data):
    print(f'Entry {i + 1}:')
    print(f'- URL:  {entry[0]}')
    print(f'- Text: {entry[1]}')


# 📃 6. Scraping many pages!

To scrape many pages, we simply add a for loop to everything...

NOAA's page numbers start at **0** in the URL:
- `https://www.noaa.gov/news?page=0` is the first page (the same listing as `/news`)
- `https://www.noaa.gov/news?page=1` is the second page
- `https://www.noaa.gov/news?page=2` is the third page

So `range(numPages)` will give us the five page numbers we want: `0, 1, 2, 3, 4`.


In [ ]:
data = []
numPages = 5

for k in range(numPages):
    # Give the URL of the page. NOAA uses 0 for its first page.
    page = f'https://www.noaa.gov/news?page={k}'
    snapshot_number = k
    src = None

    if use_saved_pages:
        snapshot_file = saved_pages_folder / f'page-{snapshot_number}.html'
        src = snapshot_file.read_bytes()
        print(f'Using saved NOAA HTML from September 28, 2026: {snapshot_file.name} (not live news)')
    else:
        # Try to get the page up to 5 times
        for i in range(5):
            try:
                response = requests.get(page, headers=my_headers, timeout=15)
                response.raise_for_status()
                if response.status_code != 200 or not response.content:
                    raise requests.HTTPError(f'Expected a nonempty HTTP 200 page; got HTTP {response.status_code}.', response=response)
                src = response.content
                break
            except requests.RequestException as error:
                print(f'Failed attempt # {i + 1}: {error}')
                # Don't retry a page that is blocked or does not exist.
                if error.response is not None and error.response.status_code in (202, 401, 403, 404):
                    break
                if i < 4:
                    time.sleep(2)


    if src:
        if not use_saved_pages:
            print(f'Successfully got page: {page}')
    else:
        raise RuntimeError(f'Could not get page: {page}. Check the error above, or set use_saved_pages = True and restart the notebook.')

    soup = BeautifulSoup(src.decode('utf-8'), 'html.parser')
    entries = soup.find_all('article', class_='node-teaser')
    if not entries:
        raise RuntimeError(f'No news articles found on {page}. Check the page before continuing.')

    for entry in entries:
        entry_text = None
        entry_url = None

        # find the title container first, then the link inside it
        found_element = entry.find('div', class_='title')
        if found_element:
            found_element = found_element.find('a')
            if found_element:
                entry_text = found_element.get_text().strip()
                href = found_element.attrs.get('href')
                if href:
                    entry_url = urljoin(page, href)

        # add all the info to the data list
        data.append([entry_url, entry_text])

    # always a good idea to take a nap
    time.sleep(2)


In [ ]:
for entry in data:
        print(entry)

# <center><font color='red'>CHALLENGE</font></center>

<div align="center">
    <img src="../assets/deal.png" width="500" alt="The class challenge">
    <br><br>
</div>

In the remainder of the class I ask that you work on the following challenge (+1% bonus)

1. Extend today's code to also scrape the "type" and "date" of each article. On NOAA, use the **Focus areas** categories as the article's "type", and the publication date in the `<time>` tag as its "date". Don't confuse Focus areas with Topics---they are different fields!
2. Save all information that you scraped about each article in a `.txt` file. Each article on the website should be a line in the file, and each article's attributes should be tab-separated, in this order: **title, URL, date, type**, like `Article_title\tArticle_url\tArticle_date\tArticle_type\n`.
3. If an article has several types, then separate the types with a dash "-", like `Article_title\tArticle_url\tArticle_date\tArticle_type1-Article_type2-Article_type3\n`. If a field is missing, leave that field blank, but keep its place in the line.
4. After you've saved the data, make sure you can read it using Python.
